In [ ]:
# Schema information: Iterate over all concepts in the DTS, extract their schema information (including hypercube membership and custom UK arcroles crossref, inflows, outflows), and write to JSON (concepts.json)

from arelle import ModelXbrl
from arelle.ModelDtsObject import ModelConcept
from arelle import XbrlConst
import json
import re
import uuid


class SimpleHypercubeFinder:
    """
    Finds all hypercubes associated with a given concept by climbing
    domain-member relationships until it finds a primary item with
    an 'all' relationship to a hypercube.
    Returns a list of hypercube QNames as strings.
    """

    def __init__(self, model_xbrl: ModelXbrl):
        self.model_xbrl = model_xbrl
        self.domainMemberRelSet = model_xbrl.relationshipSet(
            "http://xbrl.org/int/dim/arcrole/domain-member"
        )
        self.allRelSet = model_xbrl.relationshipSet(
            "http://xbrl.org/int/dim/arcrole/all"
        )

    def get_hypercubes(self, concept_ns: str, concept_name: str):
        concept = self._get_concept_by_name(concept_ns, concept_name)
        if concept is None:
            return []
        visited_concepts = set()
        found_hypercubes = set()
        self._collect_hypercubes(concept, visited_concepts, found_hypercubes)
        return sorted(str(h.qname) for h in found_hypercubes)

    def _collect_hypercubes(self, concept: ModelConcept, visited: set, results: set):
        if concept in visited:
            return
        visited.add(concept)
        parent_rels = self.domainMemberRelSet.toModelObject(concept)
        if not parent_rels:
            return
        for rel in parent_rels:
            parent = rel.fromModelObject
            if parent is not None:
                all_rels = self.allRelSet.fromModelObject(parent)
                if all_rels:
                    for all_rel in all_rels:
                        hypercube = all_rel.toModelObject
                        if hypercube is not None:
                            results.add(hypercube)
                else:
                    self._collect_hypercubes(parent, visited, results)

    def _get_concept_by_name(self, ns: str, ln: str):
        expected_prefix = None
        for qn in self.model_xbrl.qnameConcepts:
            if qn.namespaceURI.strip() == ns.strip():
                expected_prefix = getattr(qn, "prefix", str(qn).split(":")[0])
                break
        if expected_prefix is None:
            return None
        expected_qname_str = f"{expected_prefix}:{ln.strip()}"
        for qn, concept in self.model_xbrl.qnameConcepts.items():
            if str(qn).strip() == expected_qname_str:
                return concept
        return None


class ConceptDetailsExtractor:
    """
    Extracts all relevant details about a single concept from the UK Taxonomy Suite.
    """

    def __init__(self, model_taxonomy: ModelXbrl):
        self.model_taxonomy = model_taxonomy

    @staticmethod
    def is_valid_concept(concept):
        """
        Removes XBRL specific concepts (e.g. xl:documentation).
        """
        ns = concept.qname.namespaceURI
        return "lloyds" in ns and (
            concept.isItem or concept.isDimensionItem or concept.isDomainMember
        )

    def get_concept_json(self, concept_ns: str, concept_name: str):
        # Find the concept
        expected_prefix = None
        for qn in self.model_taxonomy.qnameConcepts.keys():
            if qn.namespaceURI.strip() == concept_ns.strip():
                expected_prefix = getattr(qn, "prefix", str(qn).split(":")[0])
                break
        if expected_prefix is None:
            return None
        expected_qname_str = f"{expected_prefix}:{concept_name.strip()}"
        concept = None
        for qn, c in self.model_taxonomy.qnameConcepts.items():
            if str(qn).strip() == expected_qname_str:
                concept = c
                break
        if concept is None:
            return None

        # Preferred label role (from presentation arcs)
        LABEL_ROLE_TO_TYPE = {
            "http://www.xbrl.org/2003/role/label": "Standard Label",
            "http://www.xbrl.org/2003/role/documentation": "Documentation",
            "http://www.xbrl.org/2003/role/reference": "Reference",
            "http://www.xbrl.org/2003/role/periodStartLabel": "Period Start Label",
            "http://www.xbrl.org/2003/role/periodEndLabel": "Period End Label",
            "http://www.xbrl.org/2003/role/verboseLabel": "Verbose Label",
            "http://www.xbrl.org/2003/role/terseLabel": "Terse Label",
        }
        preferred_label_role = None
        for presRel in self.model_taxonomy.relationshipSet(
            XbrlConst.parentChild
        ).toModelObject(concept):
            if getattr(presRel, "preferredLabel", None):
                role_uri = presRel.preferredLabel
                preferred_label_role = LABEL_ROLE_TO_TYPE.get(role_uri, role_uri)
                break

        # Labels
        labels = []
        for labRel in self.model_taxonomy.relationshipSet(
            XbrlConst.conceptLabel
        ).fromModelObject(concept):
            label_resource = labRel.toModelObject
            if label_resource is not None:
                role = label_resource.role
                label_type = LABEL_ROLE_TO_TYPE.get(role, role)
                labels.append(
                    {
                        "lang": label_resource.xmlLang,
                        "type": label_type,
                        "label_text": label_resource.text,
                    }
                )

        # References
        def extract_references(model_taxonomy, concept: ModelConcept):
            """
            Extract all references from the reference linkbase for this concept.
            """
            refs_info = []

            ref_rels = model_taxonomy.relationshipSet(
                XbrlConst.conceptReference
            ).fromModelObject(concept)

            for ref_rel in ref_rels:
                ref_resource = ref_rel.toModelObject
                if ref_resource is not None:
                    ref_data = {}

                    for child in ref_resource.iterchildren():
                        local_tag = (
                            child.tag.split("}")[1] if "}" in child.tag else child.tag
                        )
                        ref_data[local_tag] = child.text

                    refs_info.append(ref_data)

            return refs_info

        raw_refs = extract_references(self.model_taxonomy, concept)

        references = []
        for ref_data in raw_refs:
            lower_ref_data = {k.lower(): v for k, v in ref_data.items()}
            references.append(
                {
                    "reference_key_values": ref_data,  # full key/value mapping
                    "name": lower_ref_data.get("name"),
                    "number": lower_ref_data.get("number"),
                    "year": lower_ref_data.get("year"),
                    "schedule": lower_ref_data.get("schedule"),
                    "part": lower_ref_data.get("part"),
                    "section": lower_ref_data.get("section"),
                    "paragraph": lower_ref_data.get("paragraph"),
                    "report": lower_ref_data.get("report"),
                }
            )

        # Hypercubes
        hypercube_finder = SimpleHypercubeFinder(self.model_taxonomy)
        hypercubes = hypercube_finder.get_hypercubes(concept_ns, concept_name)

        # Arcrole extraction
        crossref_arcrole = "http://xbrl.frc.org.uk/general/types/arcroles/crossref"
        inflow_arcrole = "http://xbrl.frc.org.uk/general/types/arcroles/inflow"
        outflow_arcrole = "http://xbrl.frc.org.uk/general/types/arcroles/outflow"

        # Crossref: find all sources that point to this concept
        crossref_sources = []
        relset = self.model_taxonomy.relationshipSet(crossref_arcrole)
        if relset:
            for elr in relset.linkRoleUris:
                crossref_rels = self.model_taxonomy.relationshipSet(
                    crossref_arcrole, linkrole=elr
                ).modelRelationships
                for rel in crossref_rels:
                    if rel.fromModelObject == concept:
                        crossref_sources.append(str(rel.toModelObject.qname))
        cross_ref_destination = crossref_sources if crossref_sources else None

        # Cash flow classification: inflow/outflow if this concept is a target
        cash_flow_classification = None
        for arcrole, classification in [
            (inflow_arcrole, "inflow"),
            (outflow_arcrole, "outflow"),
        ]:
            relset = self.model_taxonomy.relationshipSet(arcrole)
            if relset is not None:
                for rel in relset.modelRelationships:
                    if rel.toModelObject == concept:
                        cash_flow_classification = classification
                        break
            if cash_flow_classification:
                break

        # Main concept dict
        concept_json = {
            "concept": {
                "local_name": concept.qname.localName,
                "xbrl_type": str(concept.baseXbrliType),
                "period_type": concept.periodType,
                "balance": concept.balance,
                "abstract": concept.isAbstract,
                "nillable": concept.nillable,
                "namespace": concept.qname.namespaceURI,
                "full_type": str(concept.typeQname) if concept.typeQname else None,
                "substitution_group": (
                    str(concept.substitutionGroupQname)
                    if concept.substitutionGroupQname
                    else None
                ),
                "preferred_label_role": preferred_label_role,
            },
            "labels": labels,
            "references": references,
            "hypercubes": hypercubes,
            "cash_flow_classification": cash_flow_classification,
            "cross_ref_destination": cross_ref_destination,
        }
        return concept_json

    def get_all_concept_details(self):
        """
        Iterate over all valid concepts and return a dictionary keyed by QName string.
        """
        all_concepts = {}
        for concept in self.model_taxonomy.qnameConcepts.values():
            if not self.is_valid_concept(concept):
                continue
            qname_str = str(concept.qname)
            ns = concept.qname.namespaceURI
            ln = concept.qname.localName
            concept_json = self.get_concept_json(ns, ln)
            if concept_json:
                all_concepts[qname_str] = concept_json
        return all_concepts

    #  1/3 Extract all hypercubes and their dimensions/primaries


def extract_hypercubes(model_xbrl):
    numeric_pattern = re.compile(r"(\d+)")
    hypercubes = {}

    all_relset = model_xbrl.relationshipSet(XbrlConst.all)
    for elr in all_relset.linkRoleUris:
        rel_set = model_xbrl.relationshipSet(XbrlConst.all, linkrole=elr)
        root_concepts = rel_set.rootConcepts

        role_type = model_xbrl.roleTypes.get(elr)
        definition = (
            role_type[0].definition if (role_type and role_type[0].definition) else elr
        )
        match = numeric_pattern.search(definition)
        elr_id = int(match.group(1)) if match else None

        for primary_item in root_concepts:
            all_relationships = rel_set.fromModelObject(primary_item)
            for all_rel in all_relationships:
                hypercube = all_rel.toModelObject
                if hypercube is None:
                    continue
                hc_qn = str(hypercube.qname)
                if hc_qn not in hypercubes:
                    # Get dimensions for this hypercube
                    dims = []
                    hc_dim_rels = model_xbrl.relationshipSet(
                        XbrlConst.hypercubeDimension, linkrole=elr
                    ).fromModelObject(hypercube)
                    for dim_rel in hc_dim_rels or []:
                        dimension = dim_rel.toModelObject
                        if dimension is not None:
                            dims.append(str(dimension.qname))
                    hypercubes[hc_qn] = {
                        "hypercube_qname": hc_qn,
                        "elr": elr,
                        "elr_id": elr_id,
                        "role_definition": definition,
                        "dimensions": dims,
                        "primary_items": set(),
                    }
                # Add this primary item to the hypercube's list
                hypercubes[hc_qn]["primary_items"].add(str(primary_item.qname))
        # Convert sets to lists for JSON serialization
    for hc in hypercubes.values():
        hc["primary_items"] = list(hc["primary_items"])
    return list(hypercubes.values())


# 2/3 Extract all dimensions, their defaults, and domain member trees


def get_default_member(model_xbrl, dimension, elr):
    rels = model_xbrl.relationshipSet(
        XbrlConst.dimensionDefault, linkrole=elr
    ).fromModelObject(dimension)
    if rels:
        for rel in rels:
            return str(rel.toModelObject.qname)
    return None


def collect_domain_tree(
    model_xbrl, dimension_concept, domain_concept, elr, visited=None
):
    if visited is None:
        visited = set()
    if domain_concept in visited:
        return None
    visited.add(domain_concept)
    children = []
    domain_member_rels = model_xbrl.relationshipSet(
        XbrlConst.domainMember, linkrole=elr
    ).fromModelObject(domain_concept)
    for rel in domain_member_rels or []:
        child = rel.toModelObject
        if child is not None:
            subtree = collect_domain_tree(
                model_xbrl, dimension_concept, child, elr, visited
            )
            if subtree:
                children.append(subtree)
    return {"member_qname": str(domain_concept.qname), "children": children}


def extract_dimensions(model_xbrl):
    numeric_pattern = re.compile(r"(\d+)")
    dimensions = []

    dim_relset = model_xbrl.relationshipSet("XBRL-dimensions")
    for elr in dim_relset.linkRoleUris:
        rel_set = model_xbrl.relationshipSet("XBRL-dimensions", linkrole=elr)
        root_concepts = rel_set.rootConcepts

        role_type = model_xbrl.roleTypes.get(elr)
        definition = (
            role_type[0].definition if role_type and role_type[0].definition else elr
        )
        match = numeric_pattern.search(definition)
        elr_id = int(match.group(1)) if match else None

        for dimension in root_concepts:
            if not (dimension.isExplicitDimension or dimension.isTypedDimension):
                continue
            default_member = get_default_member(model_xbrl, dimension, elr)
            domain_rels = model_xbrl.relationshipSet(
                XbrlConst.dimensionDomain, linkrole=elr
            ).fromModelObject(dimension)
            domain_trees = []
            for drel in domain_rels or []:
                domain = drel.toModelObject
                if domain is not None:
                    tree = collect_domain_tree(model_xbrl, dimension, domain, elr)
                    if tree:
                        domain_trees.append(tree)
            dimensions.append(
                {
                    "dimension_qname": str(dimension.qname),
                    "elr": elr,
                    "elr_id": elr_id,
                    "role_definition": definition,
                    "default_member": default_member,
                    "domain_members": domain_trees,
                }
            )
    return dimensions


# 3/3 Extract primary items and their trees for each hypercube


def is_digit_ending_hypercube_role(definition: str) -> bool:
    digit_at_end = re.compile(r".*\d+\s*$")
    return "Hypercube" in definition and bool(digit_at_end.match(definition))


def is_hypercube_elr(elr: str, definition: str) -> bool:
    """
    Returns True if the ELR or its definition indicates a hypercube.
    """
    return "Hypercube" in definition or "Hypercube" in elr


def is_hypercube(concept) -> bool:
    """
    Returns True if the concept is a hypercube (substitutionGroupQname.localName == 'hypercubeItem').
    """
    sub_group = getattr(concept, "substitutionGroupQname", None)
    return (
        sub_group is not None
        and getattr(sub_group, "localName", "").lower() == "hypercubeitem"
    )


def get_label_in_language(concept, lang_code="cy"):
    """
    Returns the label for the concept in the specified language code (e.g., 'cy' for Welsh).
    Returns None if not found.
    """
    for labRel in concept.modelXbrl.relationshipSet(
        "http://www.xbrl.org/2003/arcrole/concept-label"
    ).fromModelObject(concept):
        label_resource = labRel.toModelObject
        if (
            label_resource is not None
            and getattr(label_resource, "xmlLang", None) == lang_code
        ):
            return label_resource.text
    return None


def build_primary_items_tree(element, rel_set, visited, level=0, skip_visited=False):
    """
    Recursively build a tree of primary items (line items) for a given hypercube root.
    Excludes hypercube and dimension items from children.
    """
    if not skip_visited and element in visited:
        return None
    visited.add(element)

    # Exclude hypercube and dimension items from children
    if (
        is_hypercube(element)
        or getattr(element, "isDimensionItem", False)
        or getattr(element, "isTypedDimension", False)
    ):
        return None

    node = {
        "qname": str(element.qname),
        "label": element.label(),
        "label_cy": get_label_in_language(element, "cy"),
        "children": [],
    }

    for rel in rel_set.fromModelObject(element):
        child = rel.toModelObject
        if child is not None:
            child_node = build_primary_items_tree(
                child, rel_set, visited, level + 1, skip_visited=skip_visited
            )
            if child_node:
                node["children"].append(child_node)
    return node


def extract_hypercube_primary_items(model_taxonomy):
    """
    For each hypercube ELR, extract the hierarchical tree of primary items.
    Returns a list of dicts, one per ELR/hypercube.
    """
    numeric_pattern = re.compile(r"(\d+)")
    dim_rs = model_taxonomy.relationshipSet("XBRL-dimensions")
    elr_data = []

    for elr in dim_rs.linkRoleUris:
        rel_set = model_taxonomy.relationshipSet("XBRL-dimensions", linkrole=elr)
        role_type = model_taxonomy.roleTypes.get(elr)
        definition = (
            role_type[0].definition if (role_type and role_type[0].definition) else elr
        )
        match = numeric_pattern.search(definition)
        numeric_part = int(match.group(1)) if match else float("inf")

        # Only process if this ELR is for a hypercube
        if not is_hypercube_elr(elr, definition):
            continue

        root_concepts = rel_set.rootConcepts
        skip_visited = is_digit_ending_hypercube_role(definition)

        # Build the tree for each root concept
        visited = set()
        trees = []
        for root in root_concepts:
            tree = build_primary_items_tree(
                root, rel_set, visited, 0, skip_visited=skip_visited
            )
            if tree:
                trees.append(tree)

        elr_data.append(
            {
                "elr": elr,
                "definition": definition,
                "elr_id": numeric_part if numeric_part != float("inf") else None,
                "primary_items_tree": trees,
            }
        )

    # Sort by elr_id for consistency
    elr_data_sorted = sorted(
        elr_data, key=lambda x: x["elr_id"] if x["elr_id"] is not None else float("inf")
    )
    return elr_data_sorted


def recurse_concept(
    concept, rel_set, elr, level=0, index=0, prefix="0000", parent_rel=None
):
    tree_id = f"{prefix}" + (f"-{str(index).zfill(4)}" if level > 0 else "")
    preferred_label = parent_rel.preferredLabel if parent_rel is not None else None
    name = concept.label(preferredLabel=preferred_label, lang="en")
    return {
        "tree_id": tree_id,
        "uuid": str(uuid.uuid4()),
        "qname": str(concept.qname),
        "name": name,
        "xbrl_type": concept.typeQname.localName if concept.typeQname else "",
        "full_type": str(concept.typeQname) if concept.typeQname else "",
        "label_cy": concept.label(lang="cy") or "",
        "substitution_group": (
            str(concept.substitutionGroupQname)
            if concept.substitutionGroupQname
            else ""
        ),
        "concept_id": str(concept.qname),
        "abstract": concept.isAbstract,
        "children": [
            recurse_concept(
                rel.toModelObject, rel_set, elr, level + 1, i, tree_id, parent_rel=rel
            )
            for i, rel in enumerate(rel_set.fromModelObject(concept))
        ],
    }


def export_linkbase_tree(model_taxonomy, arcrole, filename, print_label):
    numeric_pattern = re.compile(r"(\d+)")
    tree_data = []
    for elr in model_taxonomy.relationshipSet(arcrole).linkRoleUris:
        rel_set = model_taxonomy.relationshipSet(arcrole, linkrole=elr)
        root_concepts = rel_set.rootConcepts
        role_type = model_taxonomy.roleTypes.get(elr)
        definition = (
            role_type[0].definition if role_type and role_type[0].definition else elr
        )
        numeric_match = numeric_pattern.search(definition)
        numeric_part = int(numeric_match.group(1)) if numeric_match else float("inf")
        root_tree = [
            recurse_concept(c, rel_set, elr, prefix="0000") for c in root_concepts
        ]
        tree_data.append(
            {
                "elr": elr,
                "definition": definition,
                "numeric_part": numeric_part,
                "root_tree": root_tree,
            }
        )
    tree_data.sort(key=lambda x: x["numeric_part"])
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(tree_data, f, indent=2, ensure_ascii=False)
    print(f"{print_label} exported to: {filename}")

In [ ]:
# %%
from pathlib import Path
import zipfile
import tempfile
import os
import json
from lxml import etree
from arelle import Cntlr, PackageManager

NS = {"tp": "http://xbrl.org/2016/taxonomy-package"}


def extract_taxonomy_zip(zip_path: str, extract_root: str | None = None) -> str:
    zip_path = Path(zip_path)
    if extract_root is None:
        extract_root = tempfile.mkdtemp(prefix="taxonomy_pkg_")
    extract_dir = Path(extract_root) / zip_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    return str(extract_dir)


def find_taxonomy_package_xml(extracted_dir: str) -> str:
    root = Path(extracted_dir)
    candidates = list(root.rglob("taxonomyPackage.xml"))
    if not candidates:
        top = [p.name for p in root.iterdir()] if root.exists() else []
        raise FileNotFoundError(
            f"No taxonomyPackage.xml found under: {extracted_dir}\n"
            f"Top-level contents: {top}"
        )

    # Prefer canonical META-INF location if present
    meta_inf = [
        p for p in candidates if "META-INF" in {part.upper() for part in p.parts}
    ]
    chosen = meta_inf[0] if meta_inf else candidates[0]
    return str(chosen.resolve())


def get_entrypoints_from_package(package_xml_path: str):
    tree = etree.parse(package_xml_path)
    entrypoints = tree.xpath("//tp:entryPoint", namespaces=NS)

    results = []
    package_dir = Path(package_xml_path).parent

    for ep in entrypoints:
        name = ep.findtext("tp:name", namespaces=NS) or "unnamed_entrypoint"
        ep_doc = ep.find("tp:entryPointDocument", namespaces=NS)
        href = ep_doc.get("href") if ep_doc is not None else ""

        if not href:
            print(f"[WARN] No href for entrypoint: {name}")
            continue

        # Keep remote URL as-is (Arelle package remapping should resolve it).
        # Resolve relative href to local file.
        if href.startswith(("http://", "https://")):
            entrypoint = href
        else:
            entrypoint = str((package_dir / href).resolve())

        results.append((name, entrypoint))

    return results


def run_on_entrypoint(controller, entrypoint_path, output_dir):
    print(f"  Loading: {entrypoint_path}")
    model_taxonomy = controller.modelManager.load(entrypoint_path)

    if model_taxonomy is None:
        print("    [!] Failed to load model.")
        return

    print(f"    → {len(model_taxonomy.modelXbrl.qnameConcepts)} concepts loaded.")

    # 1. Concepts
    concept_extractor = ConceptDetailsExtractor(model_taxonomy.modelXbrl)
    all_concepts = concept_extractor.get_all_concept_details()
    with open(os.path.join(output_dir, "concepts.json"), "w", encoding="utf-8") as f:
        json.dump(all_concepts, f, indent=2, ensure_ascii=False)

    # 2. Hypercubes
    hypercubes = extract_hypercubes(model_taxonomy.modelXbrl)
    with open(os.path.join(output_dir, "hypercubes.json"), "w", encoding="utf-8") as f:
        json.dump(hypercubes, f, indent=2, ensure_ascii=False)

    # 3. Dimensions
    dimensions = extract_dimensions(model_taxonomy.modelXbrl)
    with open(os.path.join(output_dir, "dimensions.json"), "w", encoding="utf-8") as f:
        json.dump(dimensions, f, indent=2, ensure_ascii=False)

    # 4. Hypercube Primary Items
    hypercube_trees = extract_hypercube_primary_items(model_taxonomy.modelXbrl)
    with open(
        os.path.join(output_dir, "primary_items.json"), "w", encoding="utf-8"
    ) as f:
        json.dump(hypercube_trees, f, indent=2, ensure_ascii=False)

    # 5. Linkbase Trees
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://www.xbrl.org/2003/arcrole/parent-child",
        os.path.join(output_dir, "presentation_tree.json"),
        "Presentation",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.org/int/dim/arcrole/hypercube-dimension",
        os.path.join(output_dir, "definition_hydim_tree.json"),
        "Definition (hycube-dim)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.org/int/dim/arcrole/dimension-domain",
        os.path.join(output_dir, "definition_dimdom_tree.json"),
        "Definition (dim-domain)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.org/int/dim/arcrole/domain-member",
        os.path.join(output_dir, "definition_dommem_tree.json"),
        "Definition (dom-member)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.org/int/dim/arcrole/dimension-default",
        os.path.join(output_dir, "definition_dimdef_tree.json"),
        "Definition (dim-default)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.org/int/dim/arcrole/all",
        os.path.join(output_dir, "definition_all_tree.json"),
        "Definition (all)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.frc.org.uk/general/types/arcroles/crossref",
        os.path.join(output_dir, "definition_crossref_tree.json"),
        "Definition (crossref)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.frc.org.uk/general/types/arcroles/inflow",
        os.path.join(output_dir, "definition_inflow_tree.json"),
        "Definition (inflow)",
    )
    export_linkbase_tree(
        model_taxonomy.modelXbrl,
        "http://xbrl.frc.org.uk/general/types/arcroles/outflow",
        os.path.join(output_dir, "definition_outflow_tree.json"),
        "Definition (outflow)",
    )


def process_taxonomy_zip(zip_path: str, output_root: str):
    controller = Cntlr.Cntlr()

    # Register ZIP as taxonomy package so Arelle can apply remappings
    PackageManager.addPackage(controller, zip_path)
    PackageManager.rebuildRemappings(controller)

    # Extract once so we can read taxonomyPackage.xml entrypoint list
    extracted_dir = extract_taxonomy_zip(zip_path)
    package_xml = find_taxonomy_package_xml(extracted_dir)
    print(f"Using taxonomyPackage.xml: {package_xml}")

    entrypoints = get_entrypoints_from_package(package_xml)
    if not entrypoints:
        print("No entrypoints found.")
        return

    os.makedirs(output_root, exist_ok=True)

    for name, entrypoint in entrypoints:
        entrypoint_folder = os.path.join(output_root, name.replace(" ", "_"))
        os.makedirs(entrypoint_folder, exist_ok=True)

        print(f"→ Generating: {name}")
        run_on_entrypoint(controller, entrypoint, entrypoint_folder)


# %%
zip_path = r"C:\Users\rmarks\Desktop\lloyds-taxonomy-2025.zip"
output_root = r"C:\Users\rmarks\Desktop\tree for lloyds"

process_taxonomy_zip(zip_path, output_root)

# %%

Using taxonomyPackage.xml: C:\Users\rmarks\AppData\Local\Temp\taxonomy_pkg_zzd_luaa\lloyds-taxonomy-2025\Lloyds-Taxonomy-2025-v2.1.0\META-INF\taxonomyPackage.xml
→ Generating: Lloyd's 2025
  Loading: https://www.lloyds.com/lloyds/2025-01-01/v2/lloyds-2025-01-01.xsd
Forbidden 
retrieving https://www.lloyds.com/lloyds/2025-01-01/v2/lloyds-2025-01-01.xsd
    → 853 concepts loaded.
Presentation exported to: C:\Users\rmarks\Desktop\tree for lloyds\Lloyd's_2025\presentation_tree.json
Definition (hycube-dim) exported to: C:\Users\rmarks\Desktop\tree for lloyds\Lloyd's_2025\definition_hydim_tree.json
Definition (dim-domain) exported to: C:\Users\rmarks\Desktop\tree for lloyds\Lloyd's_2025\definition_dimdom_tree.json
Definition (dom-member) exported to: C:\Users\rmarks\Desktop\tree for lloyds\Lloyd's_2025\definition_dommem_tree.json
Definition (dim-default) exported to: C:\Users\rmarks\Desktop\tree for lloyds\Lloyd's_2025\definition_dimdef_tree.json
Definition (all) exported to: C:\Users\rmarks